## 5. Comparativa por tipo de señal

Ahora que tenemos los sub-scores de ambos modelos, podemos hacer la comparativa
alineada con la taxonomía de anomalías de los papers originales.

Según Roy et al. (2024), DOMINANT detecta anomalías **estructurales** y **contextuales**
pero **no** joint-type. GAD-NR detecta los tres tipos. La comparativa por señal permite
responder exactamente qué aporta GAD-NR sobre DOMINANT:

| Señal | Modelo | Tipo de anomalía |
|-------|--------|------------------|
| `dom_struct` | DOMINANT | Estructural — conectividad inusual |
| `dom_attr` | DOMINANT | Contextual — features distintas a la mayoría |
| `gadnr_feat` | GAD-NR | Contextual — features distintas (señal análoga) |
| `gadnr_deg` | GAD-NR | Estructural — grado inusual (señal análoga) |
| `gadnr_h` | GAD-NR | **Joint-type** — features incompatibles con vecindario (señal nueva) |

La señal genuinamente nueva que aporta GAD-NR es `gadnr_h` — la KL divergence
de la distribución de atributos del vecindario. Es la única señal capaz de detectar
anomalías joint-type, que DOMINANT no puede detectar por diseño.

In [ ]:
# Percentiles de cada señal por tipo
pct_struct = rankdata(dom_struct)  / len(dom_struct)  * 100  # estructural (DOMINANT)
pct_attr   = rankdata(dom_attr)    / len(dom_attr)    * 100  # contextual  (DOMINANT)
pct_joint  = rankdata(gadnr_h)     / len(gadnr_h)    * 100  # joint-type  (GAD-NR)
pct_gadnr_feat = rankdata(gadnr_feat) / len(gadnr_feat) * 100  # contextual (GAD-NR)
pct_gadnr_deg  = rankdata(gadnr_deg)  / len(gadnr_deg)  * 100  # estructural (GAD-NR)

scores_df['pct_struct']      = pct_struct
scores_df['pct_attr']        = pct_attr
scores_df['pct_joint']       = pct_joint
scores_df['pct_gadnr_feat']  = pct_gadnr_feat
scores_df['pct_gadnr_deg']   = pct_gadnr_deg
scores_df['dom_score']       = dominant.decision_score_.cpu().numpy()
scores_df['gadnr_score']     = gadnr.decision_score_.cpu().numpy()

print('Percentiles calculados para las 5 señales:')
for col, label in [
    ('pct_struct',     'DOM  — estructural'),
    ('pct_attr',       'DOM  — contextual'),
    ('pct_gadnr_deg',  'GADNR — estructural (grado)'),
    ('pct_gadnr_feat', 'GADNR — contextual (features)'),
    ('pct_joint',      'GADNR — joint-type (KL vecindario)'),
]:
    p95 = np.percentile(scores_df[col], 95)
    print(f'  {label:<35}: P95 = {p95:.1f}')

### 5.1 Señales equivalentes: ¿coinciden DOMINANT y GAD-NR?

Antes de analizar la señal nueva (joint-type), comprobamos si las señales análogas
de ambos modelos coinciden en los mismos nodos:
- **Estructural:** `dom_struct` vs `gadnr_deg`
- **Contextual:** `dom_attr` vs `gadnr_feat`

Una correlación alta indicaría que ambos modelos aprenden las mismas anomalías
estructurales y contextuales. Una correlación baja indicaría que los modelos
capturan información complementaria incluso en los tipos de anomalía compartidos.

In [ ]:
# Correlación Spearman entre señales equivalentes
rho_struct, _ = spearmanr(dom_struct, gadnr_deg)
rho_attr,   _ = spearmanr(dom_attr,   gadnr_feat)
rho_joint_s, _ = spearmanr(gadnr_h, dom_struct)
rho_joint_a, _ = spearmanr(gadnr_h, dom_attr)

print('=== Correlación Spearman entre señales ===')
print(f'  Estructural : dom_struct  vs gadnr_deg  → ρ = {rho_struct:.4f}')
print(f'  Contextual  : dom_attr    vs gadnr_feat → ρ = {rho_attr:.4f}')
print(f'  Joint-type  : gadnr_h     vs dom_struct → ρ = {rho_joint_s:.4f}')
print(f'  Joint-type  : gadnr_h     vs dom_attr   → ρ = {rho_joint_a:.4f}')
print()
print('Una correlación baja entre gadnr_h y las señales de DOMINANT')
print('confirma que la señal joint-type es genuinamente nueva.')

In [ ]:
# Scatter señales equivalentes: estructural y contextual
fig, axes = plt.subplots(1, 2, figsize=FS['r1c2'])

# Panel izquierdo: señal estructural
axes[0].scatter(
    pct_struct, pct_gadnr_deg,
    s=4, alpha=0.2, color=PALETTE['neutral']
)
axes[0].plot([0, 100], [0, 100], color=PALETTE['neutral'],
             linestyle='--', linewidth=1.0, alpha=0.5)
axes[0].axvline(95, color=PALETTE['M'],  linestyle=':', linewidth=0.8, alpha=0.6)
axes[0].axhline(95, color=PALETTE['MG'], linestyle=':', linewidth=0.8, alpha=0.6)
axes[0].set_xlabel('DOM — estructural (dom_struct)',   fontsize=FONT['axis_label'])
axes[0].set_ylabel('GAD-NR — estructural (gadnr_deg)', fontsize=FONT['axis_label'])
axes[0].set_title(f'Señal estructural\nSpearman ρ = {rho_struct:.3f}',
                  fontsize=FONT['title'])
polish(axes[0], grid=False)

# Panel derecho: señal contextual
axes[1].scatter(
    pct_attr, pct_gadnr_feat,
    s=4, alpha=0.2, color=PALETTE['neutral']
)
axes[1].plot([0, 100], [0, 100], color=PALETTE['neutral'],
             linestyle='--', linewidth=1.0, alpha=0.5)
axes[1].axvline(95, color=PALETTE['M'],  linestyle=':', linewidth=0.8, alpha=0.6)
axes[1].axhline(95, color=PALETTE['MG'], linestyle=':', linewidth=0.8, alpha=0.6)
axes[1].set_xlabel('DOM — contextual (dom_attr)',       fontsize=FONT['axis_label'])
axes[1].set_ylabel('GAD-NR — contextual (gadnr_feat)', fontsize=FONT['axis_label'])
axes[1].set_title(f'Señal contextual\nSpearman ρ = {rho_attr:.3f}',
                  fontsize=FONT['title'])
polish(axes[1], grid=False)

plt.suptitle('Señales equivalentes: DOMINANT vs GAD-NR (Grafo M)',
             fontsize=FONT['title'], y=1.02)
plt.tight_layout()
save_figure(fig, '05_equivalent_signals_M.png')
plt.show()

### 5.2 Tipología de anomalías

Clasificamos cada nodo según qué señales superan el umbral P95, usando las tres
señales independientes — una por tipo de anomalía según los papers:

| Tipo | Señal | Modelo |
|------|-------|--------|
| **Estructural** | `dom_struct` ≥ P95 | DOMINANT |
| **Contextual** | `dom_attr` ≥ P95 | DOMINANT |
| **Joint-type** | `gadnr_h` ≥ P95 | GAD-NR |
| **Mixta** | dos o más señales ≥ P95 | Ambos |
| **Normal** | ninguna señal ≥ P95 | — |

In [ ]:
P_THRESH = 95

def assign_type(row):
    hs = row['pct_struct'] >= P_THRESH
    ha = row['pct_attr']   >= P_THRESH
    hj = row['pct_joint']  >= P_THRESH
    n  = sum([hs, ha, hj])
    if n >= 2:  return 'Mixta'
    elif hs:    return 'Estructural'
    elif ha:    return 'Contextual'
    elif hj:    return 'Joint-type'
    else:       return 'Normal'

scores_df['anomaly_type'] = scores_df.apply(assign_type, axis=1)

print('=== Distribución de tipos de anomalía (P95) ===')
for t, c in scores_df['anomaly_type'].value_counts().items():
    print(f'  {t:<15}: {c:5d} ({c/len(scores_df):.1%})')

In [ ]:
# Scatter tipología — dom_struct (X) vs dom_attr (Y)
# Joint-type marcados con triángulo — están en zona normal de DOMINANT
# porque precisamente DOMINANT no los detecta
TIPO_ORDER = ['Normal', 'Estructural', 'Contextual', 'Joint-type', 'Mixta']
color_map  = {
    'Normal':      PALETTE['neutral'],
    'Estructural': PALETTE['M'],
    'Contextual':  PALETTE['G'],
    'Joint-type':  PALETTE['MG'],
    'Mixta':       PALETTE['anomaly'],
}

fig, ax = plt.subplots(figsize=FS['wide'])

for atype in TIPO_ORDER:
    group  = scores_df[scores_df['anomaly_type'] == atype]
    size   = 30 if atype != 'Normal' else 8
    alpha  = 0.8 if atype != 'Normal' else 0.2
    zorder = 3 if atype != 'Normal' else 1
    marker = '^' if atype == 'Joint-type' else 'o'
    ax.scatter(
        group['pct_struct'], group['pct_attr'],
        s=size, alpha=alpha, color=color_map[atype],
        marker=marker, label=atype, zorder=zorder
    )

ax.axvline(P_THRESH, color=PALETTE['M'], linestyle='--', linewidth=1.0, alpha=0.5)
ax.axhline(P_THRESH, color=PALETTE['G'], linestyle='--', linewidth=1.0, alpha=0.5)

# Etiquetamos top-5 por score GAD-NR
top5 = scores_df.nlargest(5, 'gadnr_score')
for _, row in top5.iterrows():
    ax.annotate(
        str(row['name'])[:20],
        (row['pct_struct'], row['pct_attr']),
        fontsize=FONT['annotation'],
        xytext=(5, 3), textcoords='offset points',
        arrowprops=dict(arrowstyle='->', color='black', lw=0.5)
    )

ax.set_xlabel('Señal estructural — percentil dom_struct (DOMINANT)',
              fontsize=FONT['axis_label'])
ax.set_ylabel('Señal contextual — percentil dom_attr (DOMINANT)',
              fontsize=FONT['axis_label'])
ax.set_title('Tipología de anomalías — DOMINANT vs GAD-NR (Grafo M)',
             fontsize=FONT['title'])

legend_patches = [mpatches.Patch(color=color_map[t], label=t) for t in TIPO_ORDER]
# Añadimos el marcador triángulo para joint-type
from matplotlib.lines import Line2D
legend_patches[3] = Line2D([0], [0], marker='^', color='w',
                            markerfacecolor=PALETTE['MG'], markersize=8,
                            label='Joint-type')
ax.legend(handles=legend_patches, title='Tipo', fontsize=FONT['legend'], frameon=False)
ax.grid(alpha=0.3)
polish(ax, grid=False)
ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, '05_anomaly_typology_M.png')
plt.show()

## 6. Spotlight — candidatos del EDA

Revisamos los candidatos del EDA con las tres señales independientes.
Para cada candidato mostramos el percentil en cada señal y el tipo de anomalía
resultante según la taxonomía de los papers.

In [ ]:
def spotlight(name_fragment, df, label=None):
    """Imprime el perfil de anomalía de un candidato del EDA."""
    matches = df[df['name'].str.contains(name_fragment, case=False, na=False)]
    if len(matches) == 0:
        print(f'  [no encontrado] {name_fragment}\n')
        return

    row = matches.iloc[0]
    n   = len(df)

    def rank(col):
        return int((df[col] > row[col]).sum()) + 1

    def flag(pct):
        return '⚠' if pct >= P_THRESH else ' '

    tag = label or str(row['name'])
    print(f'  ► {tag}')
    print(f'    Ciudad: {row["city"]} ({row["state"]}) | Grado: {row["degree"]} | Tipo: {row["anomaly_type"]}')
    print(f'    DOMINANT  — estructural : #{rank("dom_struct"):5d}/{n}  ({row["pct_struct"]:.1f}p) {flag(row["pct_struct"])}')
    print(f'    DOMINANT  — contextual  : #{rank("dom_attr"):5d}/{n}  ({row["pct_attr"]:.1f}p) {flag(row["pct_attr"])}')
    print(f'    GAD-NR    — joint-type  : #{rank("gadnr_h"):5d}/{n}  ({row["pct_joint"]:.1f}p) {flag(row["pct_joint"])}')
    print()


print('=' * 70)
print('CANDIDATOS DEL EDA — TIPOLOGÍA POR SEÑAL')
print('=' * 70 + '\n')

candidatos = [
    ('Pablo',         'Pablo (alta betweenness, degree moderada)'),
    ('Shalini',       'Shalini (mayor betweenness de la red)'),
    ('Jim H',         'Jim H (super-conector + puente inter-comunidad)'),
    ('Matt Kenigson', 'Matt Kenigson (super-conector)'),
    ('GEEK',          'GEEK by AKEIN Engineering (entidad no personal)'),
]

for frag, label in candidatos:
    spotlight(frag, scores_df, label=label)

In [ ]:
# Tabla resumen ejecutivo
rows = []
for frag, label in candidatos:
    matches = scores_df[scores_df['name'].str.contains(frag, case=False, na=False)]
    if len(matches) == 0:
        continue
    row = matches.iloc[0]
    n   = len(scores_df)

    def rank(col):
        return int((scores_df[col] > row[col]).sum()) + 1

    rows.append({
        'Candidato':   label.split('(')[0].strip(),
        'Tipo':        row['anomaly_type'],
        'DOM-struct':  f'#{rank("dom_struct")} ({row["pct_struct"]:.0f}p) {"⚠" if row["pct_struct"] >= P_THRESH else ""}',
        'DOM-attr':    f'#{rank("dom_attr")} ({row["pct_attr"]:.0f}p) {"⚠" if row["pct_attr"] >= P_THRESH else ""}',
        'GADNR-joint': f'#{rank("gadnr_h")} ({row["pct_joint"]:.0f}p) {"⚠" if row["pct_joint"] >= P_THRESH else ""}',
    })

print('=== Resumen ejecutivo — candidatos del EDA ===')
print(pd.DataFrame(rows).to_string(index=False))

## 7. Comparativa DOMINANT vs GAD-NR

### 7.1 Solapamiento entre señales equivalentes

Comparamos el top-K de las señales análogas de cada modelo para verificar
si detectan los mismos nodos o poblaciones distintas.

### 7.2 Nodos exclusivos de la señal joint-type

Los nodos que superan P95 en `gadnr_h` pero no en ninguna señal de DOMINANT
son anomalías joint-type puras — el tipo que DOMINANT no puede detectar por diseño.
Su perfil (grado, comunidad, geografía) revela qué hace diferentes a estos nodos.

In [ ]:
# Solapamiento top-K entre señales equivalentes
K = 100
print(f'=== Solapamiento del top-{K} entre señales equivalentes ===')
pairs = [
    ('dom_struct', 'gadnr_deg',  'Estructural: DOM vs GAD-NR'),
    ('dom_attr',   'gadnr_feat', 'Contextual : DOM vs GAD-NR'),
]
for col1, col2, label in pairs:
    top1    = set(np.argsort(-scores_df[col1].values)[:K])
    top2    = set(np.argsort(-scores_df[col2].values)[:K])
    overlap = len(top1 & top2)
    print(f'  {label}: {overlap}/{K} nodos comunes ({overlap/K:.0%})')

# Solapamiento de joint-type con señales de DOMINANT
print()
print(f'=== Joint-type (gadnr_h) vs señales DOMINANT — top-{K} ===')
top_joint = set(np.argsort(-scores_df['gadnr_h'].values)[:K])
top_struct = set(np.argsort(-scores_df['dom_struct'].values)[:K])
top_attr   = set(np.argsort(-scores_df['dom_attr'].values)[:K])
print(f'  gadnr_h vs dom_struct: {len(top_joint & top_struct)}/{K} comunes ({len(top_joint & top_struct)/K:.0%})')
print(f'  gadnr_h vs dom_attr  : {len(top_joint & top_attr)}/{K} comunes ({len(top_joint & top_attr)/K:.0%})')
print(f'  gadnr_h vs ningún DOM: {len(top_joint - top_struct - top_attr)}/{K} nodos exclusivos joint-type')

In [ ]:
# Perfil de los nodos exclusivamente joint-type
# Nodos con pct_joint >= P95 pero pct_struct < P95 y pct_attr < P95
pure_joint = scores_df[
    (scores_df['pct_joint']  >= P_THRESH) &
    (scores_df['pct_struct'] <  P_THRESH) &
    (scores_df['pct_attr']   <  P_THRESH)
].copy()

print(f'Nodos exclusivamente joint-type: {len(pure_joint)}')
print(f'  Grado medio: {pure_joint["degree"].mean():.1f} '
      f'(vs {scores_df["degree"].mean():.1f} global)')
print()
print('Top-10 por gadnr_h:')
print(
    pure_joint.nlargest(10, 'gadnr_h')[
        ['name', 'city', 'state', 'degree', 'pct_struct', 'pct_attr', 'pct_joint']
    ].to_string(index=False)
)

In [ ]:
# Distribución de grado: joint-type vs estructural vs contextual vs normal
fig, ax = plt.subplots(figsize=FS['single'])

for atype, color in [
    ('Estructural', PALETTE['M']),
    ('Contextual',  PALETTE['G']),
    ('Joint-type',  PALETTE['MG']),
]:
    grupo = scores_df[scores_df['anomaly_type'] == atype]['degree']
    if len(grupo) == 0:
        continue
    ax.hist(grupo, bins=30, alpha=0.7, color=color,
            label=f'{atype} (n={len(grupo)})', edgecolor='white')

ax.axvline(scores_df['degree'].mean(), color=PALETTE['neutral'],
           linestyle='--', linewidth=1.0, label=f'Media global ({scores_df["degree"].mean():.0f})')

ax.set_xlabel('Grado del nodo', fontsize=FONT['axis_label'])
ax.set_ylabel('Nº de nodos',   fontsize=FONT['axis_label'])
ax.set_title('Distribución de grado por tipo de anomalía (Grafo M)',
             fontsize=FONT['title'])
ax.legend(fontsize=FONT['legend'], frameon=False)
polish(ax)
plt.tight_layout()
save_figure(fig, '05_degree_by_anomaly_type_M.png')
plt.show()

In [ ]:
# Heatmap de correlación entre todas las señales
score_cols   = ['dom_struct', 'dom_attr', 'gadnr_deg', 'gadnr_feat', 'gadnr_h']
score_labels = ['DOM-struct', 'DOM-attr', 'GADNR-deg', 'GADNR-feat', 'GADNR-KL']

n = len(score_cols)
corr_matrix = np.zeros((n, n))
for i, c1 in enumerate(score_cols):
    for j, c2 in enumerate(score_cols):
        rho, _ = spearmanr(scores_df[c1], scores_df[c2])
        corr_matrix[i, j] = rho

fig, ax = plt.subplots(figsize=FS['heatmap'])
sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap=CMAP_HEAT,
    vmin=0, vmax=1, ax=ax,
    xticklabels=score_labels, yticklabels=score_labels,
    cbar_kws={'label': 'Spearman ρ'}
)
ax.set_title('Correlación entre señales — DOMINANT vs GAD-NR (Grafo M)',
             fontsize=FONT['title'])
plt.tight_layout()
save_figure(fig, '05_signal_correlation_M.png')
plt.show()

## 8. Guardado de resultados

In [ ]:
cols_export = [
    'idx', 'member_id', 'name', 'city', 'state', 'degree',
    # Sub-scores brutos
    'dom_struct', 'dom_attr', 'dom_score',
    'gadnr_h', 'gadnr_deg', 'gadnr_feat', 'gadnr_score',
    # Percentiles por tipo de señal
    'pct_struct', 'pct_attr', 'pct_joint',
    'pct_gadnr_feat', 'pct_gadnr_deg',
    # Tipología final
    'anomaly_type',
]

scores_export = scores_df[[c for c in cols_export if c in scores_df.columns]]
scores_export.to_csv(os.path.join(RESULTS_PATH, 'scores_M.csv'), index=False)

print(f'Guardado : {RESULTS_PATH}scores_M.csv')
print(f'Shape    : {scores_export.shape}')
print(f'Columnas : {list(scores_export.columns)}')

## Resumen del notebook

En este notebook hemos:

1. **Entrenado DOMINANT** y extraído sus dos señales independientes:
   `dom_struct` (anomalía estructural) y `dom_attr` (anomalía contextual).

2. **Entrenado GAD-NR** y extraído sus tres señales: `gadnr_deg` (estructural),
   `gadnr_feat` (contextual) y `gadnr_h` (joint-type — señal genuinamente nueva).

3. **Verificado** que las señales equivalentes de ambos modelos están correlacionadas
   pero no son idénticas, y que la señal joint-type de GAD-NR es independiente de
   todas las señales de DOMINANT.

4. **Construido una tipología de 5 categorías** (Estructural, Contextual, Joint-type,
   Mixta, Normal) basada en las tres señales independientes y visualizada con el
   scatter dom_struct vs dom_attr con joint-type marcado con triángulo.

5. **Validado los candidatos del EDA** con las tres señales y caracterizado el perfil
   de los nodos exclusivamente joint-type — anomalías que DOMINANT no puede detectar.

Los resultados quedan en `data/results/scores_M.csv`.